In [ ]:
import tensorflow as tf
import sklearn
from keras.applications import EfficientNetV2B0, ResNet50, VGG16, MobileNetV2, Xception, MobileNetV3Small
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory
from keras.losses import BinaryCrossentropy
import kagglehub
import os
from datasets import load_dataset

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ds = load_dataset("ILSVRC/imagenet-1k")
pretrained_imagenet = load_dataset("timm/mini-imagenet")

In [4]:
LEARNING_RATE=0.0001
BATCH_SIZE=64
IMAGE_SIZE=160
VAL_SPLIT=0.2

In [5]:
efficientnetv2b0 = EfficientNetV2B0(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
resnet50 = ResNet50(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
vgg16 = VGG16(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
xception = Xception(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
mobilenetv3small = MobileNetV3Small(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
mobilenetv2 = MobileNetV2(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)

83683744/83683744 [==============================] - 2s 0us/step


4334752/4334752 [==============================] - 0s 0us/step


In [6]:
dataset_path = kagglehub.dataset_download("doctorstrange420/real-and-fake-ai-generated-art-images-dataset")
dataset_path

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1'

In [7]:
os.listdir(dataset_path+"\Data")

['FAKE', 'REAL']

In [8]:
fake_dir = os.path.join(dataset_path+"\Data", "FAKE")
fake_dir

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1\\Data\\FAKE'

In [9]:
real_dir = os.path.join(dataset_path+"/Data", "REAL")
real_dir

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1/Data\\REAL'

In [10]:
os.listdir(real_dir)[:5]

['00060d29813e54eec710cd6f9948a40ac.jpg',
 '000698a1cfe903e219f81178d8e7795cc.jpg',
 '00071f2b6fc24b1885ddb1b53b512081c.jpg',
 '000ad71148c3ad9bb8f68d78a0aeac70a.jpg',
 '000ad71148c3ad9bb8f68d78a0aeac70b.jpg']

In [11]:
os.listdir(fake_dir)[:5]

['img000006.jpg',
 'img000013.jpg',
 'img000014.jpg',
 'img000015.jpg',
 'img000016.jpg']

In [12]:
trainingDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="training", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
trainingDs

Found 21642 files belonging to 2 classes.
Using 17314 files for training.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [13]:
valDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="validation", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
valDs

Found 21642 files belonging to 2 classes.
Using 4328 files for validation.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [14]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1)
    ]
)
data_augmentation

In [15]:
vgg16.trainable=False

In [16]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.vgg16.preprocess_input(x)
x = vgg16.output
# x = vgg16(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)#GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [17]:
vgg16 = tf.keras.Model(inputs=vgg16.input, outputs=output)

In [18]:
vgg16.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 160, 160, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 160, 160, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 80, 80, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 80, 80, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 80, 80, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 40, 40, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 40, 40, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 40, 40, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 40, 40, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 20, 20, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 20, 20, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 20, 20, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 20, 20, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 10, 10, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 10, 10, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 10, 10, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 10, 10, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 5, 5, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,481 (56.38 MB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [19]:
vgg16.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [20]:
efficientnetv2b0.trainable=False

In [21]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.efficientnet_v2.preprocess_input(x)
x = efficientnetv2b0.output
# x = efficientnetv2b0(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [22]:
efficientnetv2b0 = tf.keras.Model(inputs=efficientnetv2b0.input, outputs=output)

In [23]:
efficientnetv2b0.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 160, 160, 3)]        0         []                            
                                                                                                  
 rescaling (Rescaling)       (None, 160, 160, 3)          0         ['input_1[0][0]']             
                                                                                                  
 normalization (Normalizati  (None, 160, 160, 3)          0         ['rescaling[0][0]']           
 on)                                                                                              
                                                                                                  
 stem_conv (Conv2D)          (None, 80, 80, 32)           864       ['normalization[0][0]'] 

In [24]:
efficientnetv2b0.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [25]:
resnet50.trainable=False

In [26]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.resnet50.preprocess_input(x)
x = resnet50.output
# x = resnet50(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [27]:
resnet50 = tf.keras.Model(inputs=resnet50.input, outputs=output)

In [28]:
resnet50.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 160, 160, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 166, 166, 3)          0         ['input_2[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 80, 80, 64)           9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 80, 80, 64)           256       ['conv1_conv[0][0]']          
 on)                                                                                        

In [29]:
resnet50.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [29]:
mobilenetv2.trainable=False

In [30]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.resnet50.preprocess_input(x)
x = mobilenetv2.output
# x = resnet50(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [31]:
mobilenetv2 = tf.keras.Model(inputs=mobilenetv2.input, outputs=output)

In [32]:
mobilenetv2.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 160, 160,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 80, 80,    │        864 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 80, 80,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 80, 80,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 80, 80,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 80, 80,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 80, 80,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 80, 80,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 80, 80,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 81, 81,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 40, 40,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 40, 40,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 40, 40,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 40, 40,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [33]:
mobilenetv2.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [ ]:
history = mobilenetv2.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=5
)

Epoch 1/5
271/271 ━━━━━━━━━━━━━━━━━━━━ 191s 673ms/step - accuracy: 0.7084 - loss: 0.5583 - precision_3: 0.7091 - recall_3: 0.7017 - val_accuracy: 0.7999 - val_loss: 0.4500 - val_precision_3: 0.8286 - val_recall_3: 0.7663
Epoch 2/5
271/271 ━━━━━━━━━━━━━━━━━━━━ 181s 670ms/step - accuracy: 0.7937 - loss: 0.4432 - precision_3: 0.7946 - recall_3: 0.7893 - val_accuracy: 0.8212 - val_loss: 0.4043 - val_precision_3: 0.8329 - val_recall_3: 0.8125
Epoch 3/5
271/271 ━━━━━━━━━━━━━━━━━━━━ 180s 663ms/step - accuracy: 0.8193 - loss: 0.4026 - precision_3: 0.8214 - recall_3: 0.8137 - val_accuracy: 0.8260 - val_loss: 0.3859 - val_precision_3: 0.8469 - val_recall_3: 0.8043
Epoch 4/5
199/271 ━━━━━━━━━━━━━━━━━━━━ 39s 542ms/step - accuracy: 0.8292 - loss: 0.3805 - precision_3: 0.8278 - recall_3: 0.8339

In [ ]:
history = efficientnetv2b0.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
)

In [ ]:
history = vgg16.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=2
)

In [ ]:
history = mobilenetv3small.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=6
    
)

In [ ]:
history = xception.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
    
)

Epoch 1/3
271/271 [==============================] - 431s 2s/step - loss: 1.6092 - accuracy: 0.5676 - precision_3: 0.5623 - recall_3: 0.5906 - val_loss: 0.6666 - val_accuracy: 0.6319 - val_precision_3: 0.6350 - val_recall_3: 0.6549
Epoch 2/3
  3/271 [..............................] - ETA: 5:45 - loss: 0.6944 - accuracy: 0.6458 - precision_3: 0.6604 - recall_3: 0.6863

In [ ]:
history = resnet50.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
    
)